<a href="https://colab.research.google.com/github/Rodrigoveloso7/otimiza-ao-agendamento-de-produ-ao/blob/main/Lista_4_Programa%C3%A7%C3%A3o_Linear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importações

In [ ]:
!apt-get install coinor-cbc

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  coinor-libcbc3 coinor-libcgl1 coinor-libclp1 coinor-libcoinutils3v5
  coinor-libosi1v5
The following NEW packages will be installed:
  coinor-cbc coinor-libcbc3 coinor-libcgl1 coinor-libclp1
  coinor-libcoinutils3v5 coinor-libosi1v5
0 upgraded, 6 newly installed, 0 to remove and 3 not upgraded.
Need to get 2,908 kB of archives.
After this operation, 8,310 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 coinor-libcoinutils3v5 amd64 2.11.4+repack1-2 [465 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 coinor-libosi1v5 amd64 0.108.6+repack1-2 [275 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 coinor-libclp1 amd64 1.17.5+repack1-1 [937 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 coinor-libcgl1 amd64 0.60.3+repack1-3 [420 kB]
Get:5 htt

In [ ]:
from pyomo.environ import *

# Resolução

Entradas

In [ ]:
custo_setor_por_produto = [[1,0.5,1.2,0.9,0.3],[20,30,18,60,80]]
capacidade_por_setor = [30,1000]
capacidade_linas_montaem = [30,100]
demanda_diaria_por_produto = [[3,0,0,5,0],[15,9,0,8,6],[0,10,0,0,11],[9,8,0,12,3],[2,0,10,0,8]]
estoques_iniciais = [[5,5,5,5,5],[5,5,5,5,5],[5,5,5,5,5]]
estoque_maximo_por_setor_final = [5,5,5]
U = [30,100]

Criando Modelo

In [ ]:
modelo = ConcreteModel()

In [ ]:
modelo.indices_produtos = range(len(demanda_diaria_por_produto[0])) # Obtendo quantos tipos de produtos existem
modelo.indices_dias = range(len(demanda_diaria_por_produto)) # Obtendo quantos dias existem

In [ ]:
modelo.corte = Var(modelo.indices_dias,modelo.indices_produtos,within=NonNegativeIntegers) # Criando variáveis de decisao quantidade cortada
modelo.pintura = Var(modelo.indices_dias,modelo.indices_produtos,within=NonNegativeIntegers) # Criando variáveis de decisao quantidade pintada
modelo.montaem = Var(modelo.indices_dias,modelo.indices_produtos,within=NonNegativeIntegers) # Criando variáveis de decisao quantidade montada
modelo.corte_estoque = Var(modelo.indices_dias,modelo.indices_produtos,within=NonNegativeIntegers) # Criando variáveis de decisao quantidade estoque cortada
modelo.pintura_estoque = Var(modelo.indices_dias,modelo.indices_produtos,within=NonNegativeIntegers) # Criando variáveis de decisao quantidade estoque pintada
modelo.montaem_estoque = Var(modelo.indices_dias,modelo.indices_produtos,within=NonNegativeIntegers) # Criando variáveis de decisao quantidade estoque montada
modelo.atraso = Var(modelo.indices_dias,modelo.indices_produtos,within=NonNegativeIntegers) # Criando variáveis de decisao quantidade acumulada de itens atrasados
modelo.avaliacao = Var(modelo.indices_dias,modelo.indices_produtos,within=Binary) # Criando variáveis de decisao que indica que somente um tipo de produto pode ser montada na lina por dia

Função Objetivo

In [ ]:
modelo.f_objetivo = Objective(expr = sum((sum(modelo.atraso[i,j]for j in modelo.indices_produtos)for i in modelo.indices_dias)),sense=minimize)
print(sum((sum(modelo.atraso[i,j]for j in modelo.indices_produtos)for i in modelo.indices_dias)))

atraso[0,0] + atraso[0,1] + atraso[0,2] + atraso[0,3] + atraso[0,4] + atraso[1,0] + atraso[1,1] + atraso[1,2] + atraso[1,3] + atraso[1,4] + atraso[2,0] + atraso[2,1] + atraso[2,2] + atraso[2,3] + atraso[2,4] + atraso[3,0] + atraso[3,1] + atraso[3,2] + atraso[3,3] + atraso[3,4] + atraso[4,0] + atraso[4,1] + atraso[4,2] + atraso[4,3] + atraso[4,4]


Restrições

In [ ]:
modelo.restr = ConstraintList()

#Restriçao capacidade de corte
for i in range(0,len(demanda_diaria_por_produto)):
  expr = sum(modelo.corte[i,j]*custo_setor_por_produto[0][j] for j in range(0,len(demanda_diaria_por_produto[0])))
  modelo.restr.add(expr <= capacidade_por_setor[0])
  print(expr <= capacidade_por_setor[0])

#Restriçao capacidade de pintura
for i in range(0,len(demanda_diaria_por_produto)):
  expr = sum(modelo.pintura[i,j]*custo_setor_por_produto[1][j] for j in range(0,len(demanda_diaria_por_produto[0])))
  modelo.restr.add(expr <= capacidade_por_setor[1])
  print(expr <= capacidade_por_setor[1])

#Restriçao de capacidade lina de montaem 1
for i in range(0,len(demanda_diaria_por_produto)):
  expr = sum(modelo.montaem[i,j] for j in range(0,2))
  modelo.restr.add(expr <= capacidade_linas_montaem[0])
  print(expr <= capacidade_linas_montaem[0])

#Restriçao que permite apenas um tipo de produto na lina 1 por dia
for i in range(len(demanda_diaria_por_produto)):
  expr = sum(modelo.avaliacao[i,j] for j in range(0,2))
  modelo.restr.add(expr <= 1)
  print(expr <= 1)


for i in range(0,len(demanda_diaria_por_produto)):
  for j in range(0,2):
    expr = modelo.montaem[i,j]
    modelo.restr.add(expr <= U[0]*modelo.avaliacao[i,j])
    print(expr <= U[0]*modelo.avaliacao[i,j])

for i in range(0,len(demanda_diaria_por_produto)):
  for j in range(2,5):
    expr = modelo.montaem[i,j]
    modelo.restr.add(expr <= U[1]*modelo.avaliacao[i,j])
    print(expr <= U[1]*modelo.avaliacao[i,j])

#Restriçao de capacidade lina de montaem 2
for i in range(0,len(demanda_diaria_por_produto)):
  expr = sum(modelo.montaem[i,j] for j in range(2,5))
  modelo.restr.add(expr <= capacidade_linas_montaem[1])
  print(expr <= capacidade_linas_montaem[1])

#Restriçao que permite apenas um tipo de produto na lina 2 por dia
for i in range(len(demanda_diaria_por_produto)):
  expr = sum(modelo.avaliacao[i,j] for j in range(2,5))
  modelo.restr.add(expr <= 1)
  print(expr <= 1)

 #---------------------------------------------------------------------------

#Restriçao de fluxo no setor de corte no primeiro dia

for j in range(0,len(demanda_diaria_por_produto[0])):
  expr = modelo.corte[0,j] - modelo.pintura[0,j] - modelo.corte_estoque[0,j]
  modelo.restr.add(expr == -estoques_iniciais[0][j])
  print(expr == -estoques_iniciais[0][j])
  modelo.restr.add(modelo.pintura[0, j]<= estoques_iniciais[0][j])

#Restriçao de fluxo no setor de corte

for i in range(1,len(demanda_diaria_por_produto)):
  for j in range(0,len(demanda_diaria_por_produto[0])):
    expr = modelo.corte[i,j] - modelo.pintura[i,j]
    modelo.restr.add(expr == modelo.corte_estoque[i,j]- modelo.corte_estoque[i-1,j])
    print(expr == modelo.corte_estoque[i,j]- modelo.corte_estoque[i-1,j])

for i in range(1, len(demanda_diaria_por_produto)):
    for j in range(len(demanda_diaria_por_produto[0])):
        modelo.restr.add( modelo.pintura[i, j]<= modelo.corte_estoque[i-1, j])
        print( modelo.pintura[i, j]<= modelo.corte_estoque[i-1, j])

#Restriçao de fluxo no setor de pintura com lina de montaem 1 no primeiro dia
for j in range(0,2):
  expr = modelo.pintura[0,j] + estoques_iniciais[1][j] - modelo.montaem[0,j]
  modelo.restr.add(expr ==  modelo.pintura_estoque[0,j])
  print(expr ==  modelo.pintura_estoque[0,j])
  modelo.restr.add(modelo.montaem[0, j]<= estoques_iniciais[1][j])

#Restriçao de fluxo no setor de pintura com lina de montaem 1

for i in range(1,len(demanda_diaria_por_produto)):
  for j in range(0,2):
    expr = modelo.pintura[i,j] + modelo.pintura_estoque[i-1,j]- modelo.montaem[i,j]
    modelo.restr.add(expr == modelo.pintura_estoque[i,j])
    print(expr == modelo.pintura_estoque[i,j])

for i in range(1, len(demanda_diaria_por_produto)):
    for j in range(0,2):
        modelo.restr.add( modelo.montaem[i, j]<= modelo.pintura_estoque[i-1, j])
        print( modelo.montaem[i, j]<= modelo.pintura_estoque[i-1, j])

#Restriçao de fluxo no setor de pintura com lina de montaem 2 no primeiro dia
for j in range(2,5):
  expr = modelo.pintura[0,j]+ estoques_iniciais[1][j] - modelo.montaem[0,j]
  modelo.restr.add(expr == modelo.pintura_estoque[0,j])
  print(expr == modelo.pintura_estoque[0,j])
  modelo.restr.add(modelo.montaem[0, j]<= estoques_iniciais[1][j])
#Restriçao de fluxo no setor de pintura com lina de montaem 2

for i in range(1,len(demanda_diaria_por_produto)):
  for j in range(2,5):
    expr = modelo.pintura[i,j] + modelo.pintura_estoque[i-1,j]- modelo.montaem[i,j]
    modelo.restr.add(expr == modelo.pintura_estoque[i,j])
    print(expr == modelo.pintura_estoque[i,j])

for i in range(1, len(demanda_diaria_por_produto)):
    for j in range(2,5):
        modelo.restr.add( modelo.montaem[i, j]<= modelo.pintura_estoque[i-1, j])
        print( modelo.montaem[i, j]<= modelo.pintura_estoque[i-1, j])

 #---------------------------------------------------------------------------------------

#Restriçao de fluxo linas de montaem primeiro dia
for j in range(0,len(demanda_diaria_por_produto[0])):
  expr = modelo.montaem[0,j] - demanda_diaria_por_produto[0][j] + modelo.atraso[0,j]
  modelo.restr.add(expr == modelo.montaem_estoque[0,j] )
  print(expr == modelo.montaem_estoque[0,j])


#Restriçao de fluxo linas de montaem

for i in range(1,len(demanda_diaria_por_produto)):
  for j in range(0,len(demanda_diaria_por_produto[0])):
    expr = modelo.montaem[i,j] + modelo.montaem_estoque[i-1,j] - demanda_diaria_por_produto[i][j] + modelo.atraso[i,j]- modelo.atraso[i-1,j]
    modelo.restr.add(expr == modelo.montaem_estoque[i,j])
    print(expr == modelo.montaem_estoque[i,j])

for j in range(0, len(demanda_diaria_por_produto[0])):
    expr = modelo.montaem_estoque[len(demanda_diaria_por_produto)-1,j]
    modelo.restr.add(expr <= estoque_maximo_por_setor_final[0])
    print(expr <= estoque_maximo_por_setor_final[0])

for j in range(0, len(demanda_diaria_por_produto[0])):
    expr = modelo.corte_estoque[len(demanda_diaria_por_produto)-1,j]
    modelo.restr.add(expr <= estoque_maximo_por_setor_final[1])
    print(expr <= estoque_maximo_por_setor_final[1])

for j in range(0, len(demanda_diaria_por_produto[0])):
    expr = modelo.pintura_estoque[len(demanda_diaria_por_produto)-1,j]
    modelo.restr.add(expr <= estoque_maximo_por_setor_final[2])
    print(expr <= estoque_maximo_por_setor_final[2])


corte[0,0] + 0.5*corte[0,1] + 1.2*corte[0,2] + 0.9*corte[0,3] + 0.3*corte[0,4]  <=  30
corte[1,0] + 0.5*corte[1,1] + 1.2*corte[1,2] + 0.9*corte[1,3] + 0.3*corte[1,4]  <=  30
corte[2,0] + 0.5*corte[2,1] + 1.2*corte[2,2] + 0.9*corte[2,3] + 0.3*corte[2,4]  <=  30
corte[3,0] + 0.5*corte[3,1] + 1.2*corte[3,2] + 0.9*corte[3,3] + 0.3*corte[3,4]  <=  30
corte[4,0] + 0.5*corte[4,1] + 1.2*corte[4,2] + 0.9*corte[4,3] + 0.3*corte[4,4]  <=  30
20*pintura[0,0] + 30*pintura[0,1] + 18*pintura[0,2] + 60*pintura[0,3] + 80*pintura[0,4]  <=  1000
20*pintura[1,0] + 30*pintura[1,1] + 18*pintura[1,2] + 60*pintura[1,3] + 80*pintura[1,4]  <=  1000
20*pintura[2,0] + 30*pintura[2,1] + 18*pintura[2,2] + 60*pintura[2,3] + 80*pintura[2,4]  <=  1000
20*pintura[3,0] + 30*pintura[3,1] + 18*pintura[3,2] + 60*pintura[3,3] + 80*pintura[3,4]  <=  1000
20*pintura[4,0] + 30*pintura[4,1] + 18*pintura[4,2] + 60*pintura[4,3] + 80*pintura[4,4]  <=  1000
montaem[0,0] + montaem[0,1]  <=  30
montaem[1,0] + montaem[1,1]  <=  30
mon

Aplicando Solver

In [ ]:
resultado = SolverFactory('cbc', executable='/usr/bin/cbc').solve(modelo,tee=True)

Welcome to the CBC MILP Solver 
Version: 2.10.7 
Build Date: Feb 14 2022 

command line - /usr/bin/cbc -printingOptions all -import /tmp/tmpyi8lstfs.pyomo.lp -stat=1 -solve -solu /tmp/tmpyi8lstfs.pyomo.soln (default strategy 1)
Option for printingOptions changed from normal to all
Presolve is modifying 20 integer bounds and re-presolving
Presolve 131 (-64) rows, 151 (-49) columns and 467 (-93) elements
Statistics for presolved model
Original problem has 200 integers (25 of which binary)
Presolved problem has 151 integers (25 of which binary)
==== 132 zero objective 3 different
1 variables have objective of -1
132 variables have objective of 0
18 variables have objective of 1
==== absolute objective values 2 different
132 variables have objective of 0
19 variables have objective of 1
==== for integers 132 zero objective 3 different
1 variables have objective of -1
132 variables have objective of 0
18 variables have objective of 1
==== for integers absolute objective values 2 different
1

In [ ]:
print('f_objetivo:',value(modelo.f_objetivo))

f_objetivo: 56.0


In [ ]:
impressao = []
for i in modelo.indices_dias:
  temp = []
  for j in modelo.indices_produtos:
    temp.append(value(modelo.montaem[i,j]))
  impressao.append(temp)
print(impressao)

[[5.0, 0.0, 0.0, 5.0, 0.0], [0.0, 10.0, 0.0, 0.0, 10.0], [22.0, 0.0, 0.0, 16.0, 0.0], [0.0, 17.0, 10.0, 0.0, 0.0], [2.0, 0.0, 0.0, 0.0, 17.0]]


In [ ]:
impressao = []
for i in modelo.indices_dias:
  temp = []
  for j in modelo.indices_produtos:
    temp.append(value(modelo.montaem_estoque[i,j]))
  impressao.append(temp)
print(impressao)

[[2.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 4.0], [9.0, 0.0, 0.0, 8.0, 0.0], [0.0, 0.0, 10.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0]]


In [ ]:
impressao = []
for i in modelo.indices_dias:
  temp = []
  for j in modelo.indices_produtos:
    temp.append(value(modelo.atraso[i,j]))
  impressao.append(temp)
print(impressao)

[[0.0, 0.0, 0.0, 0.0, 0.0], [13.0, 0.0, 0.0, 8.0, 0.0], [0.0, 9.0, 0.0, 0.0, 7.0], [0.0, 0.0, 0.0, 4.0, 10.0], [0.0, 0.0, 0.0, 4.0, 1.0]]


In [ ]:
impressao = []
for i in modelo.indices_dias:
  temp = []
  for j in modelo.indices_produtos:
    temp.append(value(modelo.corte[i,j]))
  impressao.append(temp)
print(impressao)

[[17.0, 0.0, 0.0, 11.0, 0.0], [0.0, 17.0, 0.0, 0.0, 5.0], [2.0, 0.0, 5.0, 0.0, 12.0], [0.0, 0.0, 0.0, 0.0, 5.0], [0.0, 0.0, 0.0, 0.0, 0.0]]


In [ ]:
impressao = []
for i in modelo.indices_dias:
  temp = []
  for j in modelo.indices_produtos:
    temp.append(value(modelo.corte_estoque[i,j]))
  impressao.append(temp)
print(impressao)

[[17.0, 0.0, 3.0, 11.0, 0.0], [0.0, 17.0, 3.0, 0.0, 5.0], [2.0, 0.0, 5.0, 0.0, 12.0], [0.0, 0.0, 5.0, 0.0, 5.0], [0.0, 0.0, 5.0, 0.0, 0.0]]


In [ ]:
impressao = []
for i in modelo.indices_dias:
  temp = []
  for j in modelo.indices_produtos:
    temp.append(value(modelo.pintura[i,j]))
  impressao.append(temp)
print(impressao)

[[5.0, 5.0, 2.0, 5.0, 5.0], [17.0, 0.0, 0.0, 11.0, 0.0], [0.0, 17.0, 3.0, 0.0, 5.0], [2.0, 0.0, 0.0, 0.0, 12.0], [0.0, 0.0, 0.0, 0.0, 5.0]]


In [ ]:
impressao = []
for i in modelo.indices_dias:
  temp = []
  for j in modelo.indices_produtos:
    temp.append(value(modelo.pintura_estoque[i,j]))
  impressao.append(temp)
print(impressao)

[[5.0, 10.0, 7.0, 5.0, 10.0], [22.0, 0.0, 7.0, 16.0, 0.0], [0.0, 17.0, 10.0, 0.0, 5.0], [2.0, 0.0, 0.0, 0.0, 17.0], [0.0, 0.0, 0.0, 0.0, 5.0]]


In [ ]:
impressao = []
for i in modelo.indices_dias:
  temp = []
  for j in modelo.indices_produtos:
    temp.append(value(modelo.avaliacao[i,j]))
  impressao.append(temp)
print(impressao)

[[1.0, 0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 0.0, 1.0], [1.0, 0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 1.0]]
